# Simplified Unified Embedder (Notebook)

This notebook ingests cleaned JSON chunks into Weaviate using **image + text_preview** embeddings only.

## What changed
- Uses `text_preview` from JSON directly (pre-generated, no LLM summarization at ingestion time)
- Keeps `text` for display/retrieval
- Uses `chunk_id` instead of reserved `id` property
- No CLIP token counting — `text_preview` is assumed to be within CLIP limits as generated


## Imports


In [1]:
import json
from pathlib import Path

import requests
import weaviate


## Configuration

Set your Weaviate host/ports and collection name here.


In [2]:
WEAVIATE_HOST = "172.17.0.2"
WEAVIATE_PORT = 8080
WEAVIATE_GRPC_PORT = 50051
COLLECTION_NAME = "unified_embedding"


## Schema: image + text_preview

This collection vectorizes:
- `images` (weight 0.6)
- `text_preview` (weight 0.4)

All other properties are stored but not vectorized.


In [3]:
def create_schema(delete_existing=True):
    base_url = f"http://{WEAVIATE_HOST}:{WEAVIATE_PORT}"

    if delete_existing:
        try:
            requests.delete(f"{base_url}/v1/schema/{COLLECTION_NAME}")
            print("Deleted existing collection")
        except Exception:
            pass

    schema = {
        "class": COLLECTION_NAME,
        "vectorizer": "multi2vec-clip",
        "moduleConfig": {
            "multi2vec-clip": {
                "imageFields": ["images"],
                "textFields": ["text_preview"],
                "weights": {
                    "imageFields": [0.6],
                    "textFields": [0.4],
                },
            }
        },
        "properties": [
            {
                "name": "chunk_id",
                "dataType": ["text"],
                "moduleConfig": {"multi2vec-clip": {"skip": True}},
            },
            {
                "name": "block_type",
                "dataType": ["text"],
                "moduleConfig": {"multi2vec-clip": {"skip": True}},
            },
            {
                "name": "page",
                "dataType": ["int"],
                "moduleConfig": {"multi2vec-clip": {"skip": True}},
            },
            {
                "name": "text_preview",
                "dataType": ["text"],
                "description": "Short text for embedding",
                "moduleConfig": {"multi2vec-clip": {"skip": False}},
            },
            {
                "name": "text",
                "dataType": ["text"],
                "description": "Full text (not vectorized)",
                "moduleConfig": {"multi2vec-clip": {"skip": True}},
            },
            {
                "name": "trace",
                "dataType": ["text"],
                "moduleConfig": {"multi2vec-clip": {"skip": True}},
            },
            {
                "name": "filename",
                "dataType": ["text"],
                "moduleConfig": {"multi2vec-clip": {"skip": True}},
            },
            {
                "name": "images",
                "dataType": ["blob"],
                "moduleConfig": {"multi2vec-clip": {"skip": False}},
            },
        ],
    }

    response = requests.post(
        f"{base_url}/v1/schema",
        json=schema,
        headers={"Content-Type": "application/json"},
    )

    if response.status_code == 200:
        print("Schema created")
        print("Embedding: images (0.6) + text_preview (0.4)")
        return True

    print(f"Schema creation failed: {response.status_code}")
    print(response.text)
    return False


## Ingestion Function (Single JSON)

This function:
- reads one cleaned JSON file
- uses JSON `text_preview` directly (fallback to `text`)
- clamps preview to CLIP max length only when needed
- tracks submitted vs failed writes from Weaviate batch API


In [4]:
def import_data(filepath):
    client = weaviate.connect_to_local(
        host=WEAVIATE_HOST,
        port=WEAVIATE_PORT,
        grpc_port=WEAVIATE_GRPC_PORT,
    )

    with open(filepath, "r") as f:
        data = json.load(f)

    default_filename = Path(filepath).stem.replace("_cleaned", "")
    print(f"Importing {default_filename}: {len(data)} entries")

    collection = client.collections.get(COLLECTION_NAME)
    stats = {
        "total": len(data),
        "submitted": 0,
        "success": 0,
        "failed": 0,
    }

    with collection.batch.dynamic() as batch:
        for i, entry in enumerate(data, 1):
            text = entry.get("text", "")
            text_preview = entry.get("text_preview") or text


            images = entry.get("images") or entry.get("image")
            if images == "":
                images = None

            batch.add_object(
                properties={
                    "chunk_id": entry.get("id", ""),
                    "block_type": entry.get("block_type") or entry.get("type") or "Unknown",
                    "page": entry.get("page", 0),
                    "text_preview": text_preview,
                    "text": text,
                    "trace": entry.get("trace", ""),
                    "filename": entry.get("filename", default_filename),
                    "images": images,
                }
            )
            stats["submitted"] += 1

            if i % 50 == 0:
                print(f"  Progress: {i}/{len(data)}")

    # Weaviate v4 batch errors are exposed on collection.batch.failed_objects.
    failed_objects = getattr(collection.batch, "failed_objects", []) or []
    stats["failed"] = len(failed_objects)
    stats["success"] = stats["submitted"] - stats["failed"]

    print("=" * 80)
    print(f"Submitted: {stats['submitted']}/{stats['total']}")
    print(f"Success:   {stats['success']}")
    print(f"Failed:    {stats['failed']}")
    print("=" * 80)

    client.close()
    return stats


## Run: Single File Example


In [5]:
# 1) Create / recreate schema
create_schema(delete_existing=True)

# 2) Import one cleaned JSON file
filepath = "../../clean_chunks/O-RAN-WG6.AppLCM-Deployment-R003-v02.00_cleaned.json"
stats = import_data(filepath)
stats


Deleted existing collection
Schema created
Embedding: images (0.6) + text_preview (0.4)
Importing O-RAN-WG6.AppLCM-Deployment-R003-v02.00: 376 entries
  Progress: 50/376
  Progress: 100/376
  Progress: 150/376
  Progress: 200/376
  Progress: 250/376
  Progress: 300/376
  Progress: 350/376
Submitted: 376/376
Success:   376
Failed:    0


{'total': 376, 'submitted': 376, 'success': 376, 'failed': 0}

## Batch Helper (Optional)

Use this only when you want to ingest all cleaned JSON files.


In [ ]:
def import_all_clean_chunks(data_dir="../../clean_chunks"):
    data_dir = Path(data_dir)
    json_files = sorted(data_dir.glob("*_cleaned.json"))
    print(f"Found {len(json_files)} files")

    all_stats = []
    for json_file in json_files:
        print("#" * 80)
        print(f"Processing: {json_file.name}")
        print("#" * 80)
        all_stats.append(import_data(str(json_file)))

    summary = {
        "total": sum(s["total"] for s in all_stats),
        "submitted": sum(s["submitted"] for s in all_stats),
        "success": sum(s["success"] for s in all_stats),
        "failed": sum(s["failed"] for s in all_stats),
    }

    print("=" * 80)
    print("OVERALL SUMMARY")
    print(summary)
    print("=" * 80)
    return summary


# Example:
# summary = import_all_clean_chunks("../../clean_chunks")
# summary


## Preview-Difference Review (Optional)

Shows samples where `text_preview` differs from `text`.


In [ ]:
def review_preview_examples(sample_pool=500, max_show=12):
    client = weaviate.connect_to_local(
        host=WEAVIATE_HOST, port=WEAVIATE_PORT, grpc_port=WEAVIATE_GRPC_PORT
    )

    try:
        collection = client.collections.get(COLLECTION_NAME)
        response = collection.query.fetch_objects(
            limit=sample_pool,
            return_properties=["filename", "page", "block_type", "text", "text_preview"],
        )

        changed = []
        for obj in response.objects:
            props = obj.properties or {}
            original = (props.get("text") or "").strip()
            preview = (props.get("text_preview") or "").strip()
            if original and preview and preview != original:
                changed.append(props)

        print(f"Scanned: {len(response.objects)}")
        print(f"Preview-different candidates: {len(changed)}")

        for i, props in enumerate(changed[:max_show], 1):
            original = props.get("text", "").strip()
            preview = props.get("text_preview", "").strip()
            print("=" * 80)
            print(f"Example {i}: {props.get('filename', '?')} / page {props.get('page', '?')}")
            print(f"Type: {props.get('block_type', '?')}")
            print("Original:")
            print(original[:400] + ("..." if len(original) > 400 else ""))
            print("Preview:")
            print(preview[:400] + ("..." if len(preview) > 400 else ""))

    finally:
        client.close()


# Example:
# review_preview_examples(sample_pool=500, max_show=10)


## Query Smoke Test (Optional)


In [11]:
def test_query(query="5G network architecture", limit=5):
    client = weaviate.connect_to_local(
        host=WEAVIATE_HOST, port=WEAVIATE_PORT, grpc_port=WEAVIATE_GRPC_PORT
    )

    try:
        collection = client.collections.get(COLLECTION_NAME)
        response = collection.query.near_text(
            query=query,
            limit=limit,
            return_properties=["chunk_id", "block_type", "page", "filename", "text", "text_preview"],
        )

        print(f"Found {len(response.objects)} results")
        for i, obj in enumerate(response.objects, 1):
            props = obj.properties or {}
            print("=" * 80)
            print(f"Result {i}")
            print(f"Chunk ID: {props.get('chunk_id', '')}")
            print(f"Type: {props.get('block_type', '')}")
            print(f"Page: {props.get('page', '')}")
            print(f"File: {props.get('filename', '')}")
            preview = props.get('text_preview', '') or ''
            print(f"Preview: {preview[:120]}...")

    finally:
        client.close()


# Example:
test_query("Figure 4.1-1 Common Application Life Cycle Management for ASD", limit=5)


Found 5 results
Result 1
Chunk ID: /page/0/SectionHeader/2
Type: SectionHeader
Page: 1
File: O-RAN-WG6.AppLCM-Deployment-R003-v02.00
Preview: Application Life Cycle Management \(LCM\)  for Deployment  Technical Recommendation...
Result 2
Chunk ID: /page/7/Text/4
Type: Text
Page: 8
File: O-RAN-WG6.AppLCM-Deployment-R003-v02.00
Preview: The above diagram depicts some of the key steps in basic application package orchestration:...
Result 3
Chunk ID: /page/6/Text/8
Type: Text
Page: 7
File: O-RAN-WG6.AppLCM-Deployment-R003-v02.00
Preview: The Common Application Life Cycle Management (LCM) process involves Solution Providers delivering NF deployments to Serv...
Result 4
Chunk ID: /page/28/SectionHeader/15
Type: SectionHeader
Page: 29
File: O-RAN-WG6.AppLCM-Deployment-R003-v02.00
Preview: 6.2.2 Lifecycle Requirements...
Result 5
Chunk ID: /page/9/SectionHeader/2
Type: SectionHeader
Page: 10
File: O-RAN-WG6.AppLCM-Deployment-R003-v02.00
Preview: Figure 4.2.2-1 Package Security Approaches...
